# CC3074 – Modelización y Simulación
## Flujo Máximo y Costo Mínimo

**Universidad del Valle de Guatemala**

Este notebook estudia redes donde no solamente importa encontrar un camino, sino determinar **cuánto puede transportarse** y, posteriormente, **cuánto cuesta transportarlo**.

## 1. ¿Qué es una red de flujo?

Una red de flujo puede representar:

- paquetes en una red informática,
- agua en tuberías,
- vehículos en carreteras,
- productos en una red logística,
- energía,
- información entre servidores.

Cada conexión tiene una **capacidad**, es decir, una cantidad máxima que puede atravesarla.

## 2. Símbolos fundamentales

### Fuente

$s$ = nodo desde donde sale el flujo.

### Sumidero

$t$ = nodo al que queremos llevar el flujo.

### Capacidad

$c(u,v)$ = cantidad máxima permitida desde el nodo $u$ hacia el nodo $v$.

### Flujo

$f(u,v)$ = cantidad que realmente enviamos desde $u$ hacia $v$.

Debe cumplirse:

$$
0 \leq f(u,v) \leq c(u,v)
$$

El flujo real nunca puede superar la capacidad de una conexión.

## 3. Ejemplo de Flujo Máximo

Conexiones:

```text
S → A = 4
S → B = 5
A → B = 2
A → T = 3
B → T = 4
```

`S` es la fuente y `T` es el sumidero.

Los números representan capacidades.

## 4. Resolución a mano: primer camino

Camino:

```text
S → A → T
```

Capacidades:

```text
S → A = 4
A → T = 3
```

El **cuello de botella** es la menor capacidad del camino:

$$
\min(4,3)=3
$$

Podemos enviar `3` unidades por este camino.

## 5. Resolución a mano: segundo camino

Camino:

```text
S → B → T
```

Capacidades:

```text
S → B = 5
B → T = 4
```

Cuello de botella:

$$
\min(5,4)=4
$$

Podemos enviar `4` unidades.

Flujo total:

$$
3+4=7
$$

En esta red, el flujo máximo es `7`.

## 6. Capacidad residual

La **capacidad residual** indica cuánto flujo adicional todavía puede enviarse.

Si una arista tiene capacidad `4` y ya enviamos `3`:

$$
4-3=1
$$

queda una capacidad residual de `1`.

Si usamos toda la capacidad, la capacidad residual hacia adelante queda en `0`.

## 7. ¿Por qué existe una arista residual inversa?

Cuando enviamos flujo de `u` hacia `v`, la red residual también permite representar la posibilidad de **deshacer** parte de esa decisión.

Por eso, si enviamos `x` unidades:

```text
residual[u][v] -= x
residual[v][u] += x
```

La arista inversa es importante porque un camino encontrado posteriormente puede requerir redistribuir flujo.

## 8. Crear la red residual en Python

In [ ]:
capacidades = {
    "S": {"A": 4, "B": 5},
    "A": {"B": 2, "T": 3},
    "B": {"T": 4},
    "T": {}
}

def crear_residual(capacidades):
    nodos = set(capacidades)

    for origen in capacidades:
        nodos.update(capacidades[origen])

    residual = {nodo: {} for nodo in nodos}

    for u in capacidades:
        for v, capacidad in capacidades[u].items():
            residual[u][v] = capacidad
            residual[v].setdefault(u, 0)

    return residual

residual = crear_residual(capacidades)
print(residual)

## 9. Buscar un camino aumentante

In [ ]:
def buscar_camino(residual, actual, destino, visitados, camino):
    if actual == destino:
        return camino

    visitados.add(actual)

    for vecino, capacidad in residual[actual].items():
        if capacidad > 0 and vecino not in visitados:
            resultado = buscar_camino(
                residual,
                vecino,
                destino,
                visitados,
                camino + [(actual, vecino)]
            )

            if resultado:
                return resultado

    return None

## 10. Ford-Fulkerson: Flujo Máximo

In [ ]:
def flujo_maximo(capacidades, fuente, destino):
    residual = crear_residual(capacidades)
    total = 0
    iteracion = 1

    while True:
        camino = buscar_camino(
            residual,
            fuente,
            destino,
            set(),
            []
        )

        if camino is None:
            break

        cuello = min(
            residual[u][v]
            for u, v in camino
        )

        print(
            f"Iteración {iteracion}: "
            f"{camino} | cuello={cuello}"
        )

        total += cuello

        for u, v in camino:
            residual[u][v] -= cuello
            residual[v][u] += cuello

        iteracion += 1

    return total, residual

## 11. Ejecutar Flujo Máximo

In [ ]:
resultado, residual_final = flujo_maximo(
    capacidades,
    "S",
    "T"
)

print("\nFlujo máximo:", resultado)

### Resultado

```text
Flujo máximo: 7
```

La implementación utiliza caminos aumentantes hasta que ya no existe un camino desde `S` hasta `T` con capacidad residual positiva.

# Parte II – Costo Mínimo

Ahora cada conexión tendrá dos propiedades:

1. **Capacidad:** cuánto puede transportar.
2. **Costo:** cuánto cuesta transportar una unidad.

Ejemplo:

```text
S → A
Capacidad = 4
Costo = 2 por unidad
```

Si enviamos `3` unidades:

$$
3 \times 2 = 6
$$

El costo de esa asignación es `6`.

## 12. Costo de un camino

Si un camino utiliza varias aristas, primero sumamos sus costos por unidad.

Por ejemplo:

```text
S → A → T
```

Si:

```text
S → A cuesta 2 por unidad
A → T cuesta 2 por unidad
```

entonces una unidad enviada por ese camino cuesta:

$$
2+2=4
$$

Si enviamos `3` unidades:

$$
3 \times 4=12
$$

## 13. Ejemplo integrador de la presentación

Red:

```text
S → A : capacidad 4, costo 2
S → B : capacidad 3, costo 1
A → T : capacidad 3, costo 2
B → T : capacidad 3, costo 3
A → B : capacidad 2, costo 1
```

Caminos simples relevantes:

```text
S → A → T
S → B → T
S → A → B → T
```

Costos por unidad:

```text
S → A → T       = 2 + 2     = 4
S → B → T       = 1 + 3     = 4
S → A → B → T   = 2 + 1 + 3 = 6
```

Los dos primeros caminos son más baratos por unidad que el tercero.

## 14. Ejemplo a mano: enviar 5 unidades al menor costo

Podemos utilizar:

```text
S → A → T
```

con hasta `3` unidades por la capacidad de `A → T`.

Costo:

$$
3 \times 4=12
$$

Todavía necesitamos enviar:

$$
5-3=2
$$

Por:

```text
S → B → T
```

enviamos `2` unidades.

Costo:

$$
2 \times 4=8
$$

Costo total:

$$
12+8=20
$$

En esta red particular, para enviar 5 unidades podemos hacerlo con costo total `20` utilizando esos dos caminos.

## 15. Implementación didáctica de Flujo de Costo Mínimo

Para resolver el problema de forma general utilizaremos una red residual con:

- capacidad residual,
- costo hacia adelante,
- costo negativo en la arista inversa.

En cada iteración buscaremos un camino de costo mínimo en la red residual usando **Bellman-Ford**.

Bellman-Ford es conveniente aquí porque las aristas residuales inversas pueden tener costos negativos.

In [ ]:
def crear_red_costo(aristas):
    red = {}

    for u, v, capacidad, costo in aristas:
        red.setdefault(u, [])
        red.setdefault(v, [])

        # Índices de las aristas inversas.
        indice_uv = len(red[v])
        indice_vu = len(red[u])

        red[u].append({
            "to": v,
            "cap": capacidad,
            "cost": costo,
            "rev": indice_uv
        })

        red[v].append({
            "to": u,
            "cap": 0,
            "cost": -costo,
            "rev": indice_vu
        })

    return red

In [ ]:
def camino_menor_costo(red, fuente, destino):
    nodos = list(red.keys())
    infinito = float("inf")

    distancia = {n: infinito for n in nodos}
    anterior = {n: None for n in nodos}

    distancia[fuente] = 0

    # Relajación Bellman-Ford.
    for _ in range(len(nodos) - 1):
        cambio = False

        for u in nodos:
            if distancia[u] == infinito:
                continue

            for i, arista in enumerate(red[u]):
                if arista["cap"] <= 0:
                    continue

                v = arista["to"]
                nueva = distancia[u] + arista["cost"]

                if nueva < distancia[v]:
                    distancia[v] = nueva
                    anterior[v] = (u, i)
                    cambio = True

        if not cambio:
            break

    if distancia[destino] == infinito:
        return None, None

    camino = []
    actual = destino

    while actual != fuente:
        u, indice = anterior[actual]
        camino.append((u, indice))
        actual = u

    camino.reverse()
    return camino, distancia[destino]

In [ ]:
def flujo_costo_minimo(aristas, fuente, destino, flujo_requerido):
    red = crear_red_costo(aristas)

    flujo_total = 0
    costo_total = 0
    iteracion = 1

    while flujo_total < flujo_requerido:
        camino, costo_unitario = camino_menor_costo(
            red,
            fuente,
            destino
        )

        if camino is None:
            break

        disponible = flujo_requerido - flujo_total

        cuello = min(
            red[u][indice]["cap"]
            for u, indice in camino
        )

        enviar = min(disponible, cuello)

        ruta = [fuente]

        for u, indice in camino:
            arista = red[u][indice]
            v = arista["to"]

            arista["cap"] -= enviar
            red[v][arista["rev"]]["cap"] += enviar

            ruta.append(v)

        flujo_total += enviar
        costo_total += enviar * costo_unitario

        print(
            f"Iteración {iteracion}: "
            f"{' → '.join(ruta)} | "
            f"flujo={enviar} | "
            f"costo/unidad={costo_unitario}"
        )

        iteracion += 1

    return flujo_total, costo_total

## 16. Ejecutar el ejemplo de Costo Mínimo

In [ ]:
aristas = [
    ("S", "A", 4, 2),
    ("S", "B", 3, 1),
    ("A", "T", 3, 2),
    ("B", "T", 3, 3),
    ("A", "B", 2, 1)
]

flujo, costo = flujo_costo_minimo(
    aristas,
    "S",
    "T",
    flujo_requerido=5
)

print("\nFlujo enviado:", flujo)
print("Costo total:", costo)

### Resultado esperado

```text
Flujo enviado: 5
Costo total: 20
```

Interpretación: fue posible enviar las 5 unidades solicitadas respetando las capacidades y obteniendo un costo total de 20 para este ejemplo.

## 17. Diferencia entre los dos problemas

### Flujo Máximo

Pregunta:

> ¿Cuál es la mayor cantidad que puedo transportar?

Se concentra principalmente en las **capacidades**.

### Flujo de Costo Mínimo

Pregunta:

> Para una cantidad de flujo requerida, ¿cómo puedo transportarla con el menor costo?

Se consideran simultáneamente:

- capacidades,
- costos,
- cantidad requerida.

## 18. Conexión con los otros temas

### A*
Busca un camino utilizando:

$$
f(n)=g(n)+h(n)
$$

### Hill Climbing
Busca mejorar una solución utilizando vecinos.

### Flujo Máximo
Busca cuánto puede transportarse a través de una red.

### Costo Mínimo
Busca cómo transportar una cantidad requerida minimizando el costo.

La elección del algoritmo depende de **qué pregunta queremos responder**.

## 19. Ejercicio

Utilice la red:

```text
S → A : capacidad 5, costo 1
S → B : capacidad 4, costo 3
A → B : capacidad 2, costo 1
A → T : capacidad 3, costo 4
B → T : capacidad 5, costo 1
```

Realice:

1. Dibuje la red.
2. Identifique fuente y sumidero.
3. Calcule caminos y cuellos de botella.
4. Determine el flujo máximo.
5. Solicite un flujo de 5 unidades.
6. Determine manualmente una distribución de costo mínimo.
7. Compruebe el resultado con Python.

## 20. Idea final

En una red debemos distinguir claramente:

```text
capacidad = cuánto PUEDE pasar
flujo     = cuánto ESTÁ pasando
costo     = cuánto cuesta cada unidad enviada
```

Esta diferencia permite modelar redes de transporte, comunicaciones, logística y muchos otros sistemas.